## 1. Preprocesamiente del corpus Medline

In [ ]:
%pip install nltk

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

In [ ]:
from pathlib import Path
import os

# Obtener el directorio actual y construir la ruta al archivo
currentDirectory = Path().resolve()
file_path = os.path.join(currentDirectory, "med", "MED.ALL")
query_path = os.path.join(currentDirectory, "med", "MED.QRY")

def extract_documents(file_path):
    docs_raw = []
    current_doc = []
    inside_text = False

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # Detecta inicio de nuevo documento
            if line.startswith(".I "):
                # Si ya había texto acumulado, lo guardamos como documento previo
                if current_doc:
                    docs_raw.append(" ".join(current_doc).strip())
                    current_doc = []
                inside_text = False  # aún no estamos en la parte .W

            # Detecta comienzo del texto
            elif line == ".W":
                inside_text = True

            # Si estamos dentro del texto, acumulamos contenido
            elif inside_text:
                if line != "":
                    current_doc.append(line)

        # guardar último documento si quedó en buffer
        if current_doc:
            docs_raw.append(" ".join(current_doc).strip())

    return docs_raw

docs_raw = extract_documents(file_path)
queries_raw = extract_documents(query_path)

print(f"Total de documentos cargados: {len(docs_raw)}")

print(f"\nTotal de consultas cargadas: {len(queries_raw)}")

# Nos quedamos solo con las 10 primeras consultas:
queries_raw = queries_raw[:10]

In [ ]:
import re
def preprocess(text):
    """
    1. Conversión a minúsculas (lowercasing)
    2. Tokenización (separación de palabras)
    3. Eliminación de caracteres no alfabéticos
    4. Eliminación de stopwords
    5. Stemming (raíces morfológicas)
    """
    # 1. Lowercasing
    text = text.lower()
    # 2. Eliminación de caracteres no alfanuméricos
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    # 3. Tokenización
    tokens = text.split()
    # 4. Eliminación de caracteres no alfabéticos
    tokens = [t for t in tokens if t.isalpha()]
    # 5. Eliminación de stopwords
    stopwords = nltk.corpus.stopwords.words("english")
    tokens = [t for t in tokens if t not in stopwords]
    # 6. Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

corpus_tokens = [preprocess(doc) for doc in docs_raw]
queries_tokens = [preprocess(query) for query in queries_raw]

print(f"Tokens del primer documento:\n{corpus_tokens[0][:20]}")

In [ ]:
from collections import Counter
doc_tf = [Counter(doc) for doc in corpus_tokens]
query_tf = [Counter(query) for query in queries_tokens]
doc_len = [len(doc) for doc in corpus_tokens]
query_len = [len(query) for query in queries_tokens]

## 2. Construcción del modelo de lenguaje del documento

In [ ]:
# Calcular el vocabulario único de la colección
vocabulary = set()
for doc in corpus_tokens:
    vocabulary.update(doc) # update hace la unión entra la lista y el set
V = len(vocabulary)

# Calcular frecuencias totales en la colección (para P(w|C))
collection_tf = Counter()
for tf_dict in doc_tf:
    collection_tf.update(tf_dict)

# Suma total de términos en la colección
total_terms_in_collection = sum(doc_len)
mu = total_terms_in_collection / len(corpus_tokens)

print(f"Tamaño del vocabulario: {V}")
print(f"Total de términos en la colección: {total_terms_in_collection}")

def prob_term_in_collection(term):
    """
    P(w|C) - Probabilidad del término en la colección completa
    """
    if term not in collection_tf:
        return 1e-10  
    return collection_tf[term] / total_terms_in_collection

def language_model_laplace(doc_tf, doc_len, term, V, alpha=1):
    """
    Suavizado de Laplace (add-one smoothing)
    P(w|d) = (tf(w,d) + α) / (|d| + V * α)
    
    Args:
        doc_tf: Counter con frecuencias de términos del documento
        doc_len: longitud del documento
        term: término a calcular probabilidad
        V: tamaño del vocabulario
    """
    return (doc_tf[term] + alpha) / (doc_len + V * alpha)

def language_model_jelinek_mercer(doc_tf, doc_len, term, lambda_param=0.5):
    """
    Suavizado de Jelinek-Mercer 
    P(w|d) = λ * (tf(w,d) / |d|) + (1-λ) * P(w|C)
    P(w|d) = λ · P(w | Md ) + (1 - λ) · P(w | MD )
    
    Args:
        doc_tf: Counter con frecuencias de términos del documento
        doc_len: longitud del documento
        term: término a calcular probabilidad
        lambda_param: alto -> más peso al documento, más bajo -> más peso al corpus (entre 0 y 1)
    """
    p_md = doc_tf[term] / doc_len if doc_len > 0 else 0  
    p_c = prob_term_in_collection(term)  # Probabilidad en colección
    return lambda_param * p_md + (1 - lambda_param) * p_c

def language_model_dirichlet(doc_tf, doc_len, term, mu=mu):
    """
    Suavizado de Dirichlet
    P(w|d) = (|d| / (|d| + μ)) · P(w | Md ) + (μ / (|d| + μ)) · P(w | MD )
    λ = |d| / (|d| + μ)
    1 - λ = μ / (|d| + μ)
    Args:
        doc_tf: Counter con frecuencias de términos del documento
        doc_len: longitud del documento
        term: término a calcular probabilidad
        mu: longitud promedio del los documentos de la colección (μ)
    """
    lambda_param1 = doc_len / (doc_len + mu)
    lambda_param2 = mu / (doc_len + mu)
    p_md = doc_tf[term] / doc_len if doc_len > 0 else 0  
    p_c = prob_term_in_collection(term)  # Probabilidad en colección
    return lambda_param1 * p_md + lambda_param2 * p_c

## 3. Implementación del modelo Query Likelihood

In [ ]:
import math

def query_likelihood(query_tokens, smoothing_method='dirichlet', **params):
    """
    Calcula P(q|d) para todos los documentos usando el modelo Query Likelihood.
    
    Args:
        query_tokens: lista de tokens de la consulta (ya preprocesados)
        smoothing_method: método de suavizado ('laplace', 'jelinek_mercer', 'dirichlet')
        **params: parámetros adicionales para cada método
            - alpha: para Laplace (default=1)
            - lambda_param: para Jelinek-Mercer (default=0.7)
            - mu: para Dirichlet (default=longitud_promedio)
    
    Returns:
        Lista de tuplas (score, doc_id) ordenadas por score descendente
    """
    
    # Seleccionar función de suavizado
    if smoothing_method == 'laplace':
        alpha = params.get('alpha', 1)
        smooth_func = lambda dtf, dlen, term: language_model_laplace(dtf, dlen, term, V, alpha)
    elif smoothing_method == 'jelinek_mercer':
        lambda_param = params.get('lambda_param', 0.7)
        smooth_func = lambda dtf, dlen, term: language_model_jelinek_mercer(dtf, dlen, term, lambda_param)
    elif smoothing_method == 'dirichlet':
        mu_param = params.get('mu', mu)
        smooth_func = lambda dtf, dlen, term: language_model_dirichlet(dtf, dlen, term, mu_param)
    else:
        raise ValueError(f"Método de suavizado '{smoothing_method}' no reconocido")
    
    scores = []
    
    # Calcular log P(q|d) para cada documento
    for doc_idx in range(len(doc_tf)):
        log_prob = 0.0
        
        for term in query_tokens:
            log_prob += math.log(smooth_func(doc_tf[doc_idx], doc_len[doc_idx], term))
        
        scores.append((log_prob, doc_idx))
    
    # Ordenar por score descendente
    scores.sort(reverse=True, key=lambda x: x[0])
    
    return scores

## 4. Ejecución de consultas

In [ ]:
%pip install prettytable

In [ ]:
from prettytable import PrettyTable

suavizados = ['laplace', 'jelinek_mercer', 'dirichlet']
resultados_por_query = []
for query_idx in range(len(queries_tokens)):
    print(f"\nQuery {query_idx + 1}: {queries_tokens[query_idx]}")
    table = PrettyTable()
    table.field_names = ["Rank"] + suavizados
    resultados_por_suavizado = {s: query_likelihood(queries_tokens[query_idx], smoothing_method=s) for s in suavizados} 
    for rank in range(10):
        row = [rank + 1]
        for s in suavizados:
            score, doc_id = resultados_por_suavizado[s][rank]
            row.append(f"Doc {doc_id:4d} | Score: {score:.4f}")
        table.add_row(row)
    resultados_por_query.append(resultados_por_suavizado)
    print(table)

## 5. Evaluación de resultados con MED.REL

Cada línea de MED.REL tiene el formato:

<query_id> 0 <doc_id> <relevance_score>.


<relevance_score> es siempre uno, por lo que lo ignoramos

In [ ]:
# Cargar documentos relevantes de MED.REL
rel_path = os.path.join(currentDirectory, "med", "MED.REL")

# Estructura: {query_id: [list of relevant doc_ids]}
relevance_judgments = {}

with open(rel_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        query_id = int(parts[0])
        doc_id = int(parts[2]) - 1  # Convertir a índice (empiezan en 1)
        
        if query_id not in relevance_judgments:
            relevance_judgments[query_id] = []
        relevance_judgments[query_id].append(doc_id)

from prettytable import PrettyTable

for query_idx in range(len(queries_tokens)):
    query_id = query_idx + 1
    
    relevant_docs = relevance_judgments[query_id]
    
    print(f"\nQuery {query_id}: {len(relevant_docs)} documentos relevantes en total")
    
    table = PrettyTable()
    table.field_names = ["Método", "Cuántos son relevantes?", "Precisión"]
    
    for suavizado in suavizados:
        results = resultados_por_query[query_idx][suavizado]
        top10_docs = [doc_id for score, doc_id in results[:10]]
        
        # Contar cuántos de los top-10 son relevantes
        relevant_in_top10 = len([d for d in top10_docs if d in relevant_docs])
        precision_at_10 = relevant_in_top10 / 10.0
        
        table.add_row([suavizado, f"{relevant_in_top10}/10", f"{precision_at_10:.2%}"])
    
    print(table)

# Resumen general
print()
print("RESUMEN: Precisión promedio por método")
print()

summary_table = PrettyTable()
summary_table.field_names = ["Método", "Promedio"]

for suavizado in suavizados:
    total_precision = 0.0
    num_queries = 0
    
    for query_idx in range(len(queries_tokens)):
        query_id = query_idx + 1
        
        if query_id not in relevance_judgments:
            continue
        
        relevant_docs = set(relevance_judgments[query_id])
        results = resultados_por_query[query_idx][suavizado]
        top10_docs = [doc_id for score, doc_id in results[:10]]
        
        relevant_in_top10 = len([d for d in top10_docs if d in relevant_docs])
        total_precision += relevant_in_top10 / 10.0
        num_queries += 1
    
    avg_precision = total_precision / num_queries if num_queries > 0 else 0
    summary_table.add_row([suavizado, f"{avg_precision:.1%}"])

print(summary_table)

## 6. Conclusiones

### Métodos de suavizado implementados

Se implementaron tres técnicas de suavizado para modelos de lenguaje aplicados al corpus MEDLINE (1033 documentos, vocabulario V=8,870):

**Laplace**: Añade una constante α=1 a todas las frecuencias.

**Jelinek-Mercer**: Interpolación entre probabilidades del documento y la colección con λ=0.7, lo cual le da una mayor confianza al documento sobre el corpues global. 

**Dirichlet**: Adapta el suavizado según la longitud del documento. Esto hace que en la práctica sea el mejor método. Lo cual queda reflejado en los resultados obtenidos que explicaremos a continuación.

Para comparar los métods, se comparan los documentos recuperados juntos son scores para las 10 primeras consultas del conjunto MED.QRY. Además, dado que tampoco nos aporta demasiada información, hemos optado por emplear MED.REL para calcular la precisión de cada método para cada consulta.

### Análisis de resultados

Los tres métodos obtienen Precisión promedio similar (63-64%), lo que indica robustez del modelo Query Likelihood independientemente del suavizado elegido. Sin embargo, se observan diferencias a nivel individual:

Las queries con mayor número de documentos relevantes (Query 5: 26 docs, Query 9: 28 docs) muestran mayor discriminación entre métodos. Jelinek-Mercer y Dirichlet logran 100% de precisión en Query 5, mientras Laplace alcanza solo 80%. En Query 9, Dirichlet (60%) supera a Laplace (40%) y Jelinek-Mercer (50%). Esto también nos muestra que aunque haya una gran cantidad de documentos relevantes, el método de suavizado puede influir significativamente en la precisión obtenida y que mayor número de documentos relevantes no implica necesariamente mayor precisión.

Aún así, queries con pocos documentos relevantes (Query 6: 13 docs, Query 8: 11 docs) presentan precisiones más bajas y mayor variabilidad. En Query 8, Dirichlet (50%) supera significativamente a Jelinek-Mercer (30%).

- **Dirichlet**: Mejor rendimiento en consultas complejas (Query 8, Query 9) gracias a su adaptación automática según longitud del documento.

La similitud en los promedios (±1%) confirma que el modelo Query Likelihood es robusto, pero Dirichlet ofrece ventajas teóricas y mejores resultados en casos desafiantes, justificando su recomendación para colecciones de documentos cortos como MEDLINE.